# Exercise — Interpret Quality Results & Plan Remediation

Run **Trailhead Provisions**' quality suite (Great-Expectations-shaped results across six
dimensions). For each failure, choose a remediation strategy — **source**, **downstream**, or
**quarantine** — and write a prioritized plan. See `INSTRUCTIONS.md`.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import GovernedCatalog

gc = GovernedCatalog("trailhead.db")
q = gc.run_quality_suite()
print("Failing expectations:", int((~q["success"]).sum()), "of", len(q))
q

In [ ]:
q.groupby("dimension").agg(checks=("success", "size"),
                           failing=("success", lambda s: int((~s).sum())),
                           worst_pct=("unexpected_percent", "max")).reset_index()

## 1. Classify each failing expectation
Map every `table.column` failure to `source` | `downstream` | `quarantine`.

In [ ]:
remediation = {
    "product.category":             "source",       # producer must populate mandatory attr
    "product.sustainability_class": "source",       # CSRD attr; fix at inventory domain
    "customer.email":               "quarantine",   # invalid PII contact; isolate from sends
    "customer.phone":               "quarantine",   # invalid contact; isolate
    "orders.amount":                "source",        # negative amounts = upstream error
    "shipment.carbon_kg":           "downstream",    # unit normalization in the transform
    "customer.created_ts":          "downstream",    # backfill/flag stale at load
    "campaign_membership.mkt_consent": "source",     # consent must come from the SoR
}
fails = q[~q["success"]].assign(key=lambda d: d["table"] + "." + d["column"])
fails["strategy"] = fails["key"].map(remediation)
fails[["key", "dimension", "unexpected_percent", "strategy"]]

## 2. Prioritized remediation plan
Replace the cell below with your plan.

### Prioritized remediation plan

**Priority 1 — quarantine now (GDPR + active harm):** `customer.email` and `customer.phone`
validity failures. Invalid contact data on PII columns means sends fail *and* a subject-access
request returns garbage. Quarantine these records out of marketing audiences until corrected.

**Priority 2 — fix at source:** `product.category`, `product.sustainability_class` (CSRD attr,
~50% missing), `orders.amount` (negatives are an integrity error), and
`campaign_membership.mkt_consent` (must be read from the loyalty consent SoR, not captured
independently). Tie to contract SLOs: producers meet completeness ≥ 0.99 before publish.

**Priority 3 — downstream:** `shipment.carbon_kg` unit normalization belongs in the reporting
transform; `customer.created_ts` staleness can be flagged at load.

**Drift watch:** the consent mismatch (consistency, ~37%) and carbon unit (accuracy) recur every
load — assert them as ongoing SLOs and feed them to the governance dashboard.